# 02 — Preprocessing (Data cleaning)

Combines the train and dev splits, validates the schema with `pandera`, then applies domain-aware cleaning: URLs are stripped and their registrable domains (e.g. `cnn.com`) kept as an auxiliary feature column.

The same cleaning logic is available for production use in `src/data/clean_domain_aware.py`.

## Imports

In [ ]:
!pip install nltk
!pip show nltk
!pip install tldextract
!pip install datasets
!pip install shap

In [ ]:
import os
import requests
import json
import time
import zipfile

import nltk
import kagglehub
import re

import matplotlib as mpl
import matplotlib.pyplot as plt

import seaborn as sns
import pandas as pd
import numpy as np

from collections import Counter

import tldextract

from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer


nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')

from sklearn.preprocessing import LabelEncoder, MultiLabelBinarizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_recall_fscore_support
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from scipy.sparse import hstack
from sklearn.svm import SVC

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, Callback
from tensorflow.keras.mixed_precision import set_global_policy
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Conv1D, GlobalMaxPooling1D, LSTM, Dense, SpatialDropout1D, Dropout, BatchNormalization, Bidirectional, Attention
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from keras.metrics import AUC, Precision, Recall

import torch
from torch.utils.data import DataLoader
from torch.nn import CrossEntropyLoss
from torch.optim import AdamW
from tensorflow.keras.optimizers import Adam
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset

from imblearn.over_sampling import SMOTE
import shap

## Getting the data

In [ ]:
# Download latest version
path = kagglehub.dataset_download("aadyasingh55/sexism-detection-in-english-texts")


print("Path to dataset files:", path)

dev_df = pd.read_csv(f"{path}/dev.csv")
test_df = pd.read_csv(f"{path}/test (1).csv")
train_df = pd.read_csv(f"{path}/train (2).csv")


## Schema validation

In [ ]:
import pandera.pandas as pa

schema = pa.DataFrameSchema({
     'rewire_id': pa.Column(str),
     'text': pa.Column(str),
     'label_sexist': pa.Column(str),
     'label_category': pa.Column(str),
     'label_vector': pa.Column(str),
     'split': pa.Column(str),
 })

In [ ]:
schema.validate(train_df)
schema.validate(test_df)
schema.validate(dev_df)

###Combining train and dev

In [ ]:
train_dev_data= pd.concat([dev_df, train_df], ignore_index=True) # Combine datasets
train_dev_data= train_dev_data.sample(frac=1, random_state=1).reset_index(drop=True) # Shuffle the new dataset

train_dev_data['label_sexist'].value_counts()

In [ ]:
train_dev_data.info()

In [ ]:
train_dev_data.describe()

In [ ]:
print(train_dev_data['label_category'].unique())

In [ ]:
# Check for duplicate rows based on 'rewire_id' and 'text'
duplicate_entries = train_dev_data[train_dev_data.duplicated(subset=["rewire_id", "text"])]

# Display results
print(f"Number of duplicate rows based on 'rewire_id' and 'text': {duplicate_entries.shape[0]}")
print(duplicate_entries)

## Lemmatisation & domain-aware cleaning

`clean_text` returns the cleaned text **and** the list of URL domains found in it.

In [ ]:
# Initialise NLP tools
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

# Function to clean text and extract domains
def clean_text(text, lowercase=True, replace_urls=True, extract_domain=False, remove_stopwords=True, lemmatize=True):
    if lowercase:
        text = text.lower()

    # Extract domain names from URLs
    url_pattern = r'https?://\S+|www\.\S+'
    domains = []  # Store extracted domains
    matches = re.findall(url_pattern, text)

    for match in matches:
        extracted = tldextract.extract(match)
        domain = f"{extracted.domain}.{extracted.suffix}"  # e.g., "cnn.com"
        domains.append(domain)  # Save domain for analysis
        text = text.replace(match, "")  # Remove the URL from text

    # Remove special characters, punctuation, and numbers
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    # Tokenization
    tokens = word_tokenize(text)

    # Remove stopwords
    if remove_stopwords:
        tokens = [word for word in tokens if word not in stop_words]

    # Lemmatization
    if lemmatize:
        tokens = [lemmatizer.lemmatize(word) for word in tokens]

    return " ".join(tokens), domains  # Convert tokens back to string


In [ ]:
# Apply function to dataset
train_dev_data[["text", "domains"]] = pd.DataFrame(train_dev_data["text"].apply(clean_text).tolist(), index=train_dev_data.index)

# Process the separate test set as well
test_df[["text", "domains"]] = pd.DataFrame(test_df["text"].apply(clean_text).tolist(), index=test_df.index)

print(train_dev_data.head())


In [ ]:
# Check which unique domains were extracted
unique_domains = pd.Series([domain for domain_list in train_dev_data['domains'] for domain in domain_list]).value_counts()
print(unique_domains.head(10))  # Display top 10 most frequent domains